# D65-FairFace7-ROI Walkthrough
**Chart-free flash/no-flash cheek colorimetry with FairFace-routed sampling**

FitSkin / Pansor · August 2026

---

## How to run in Google Colab

1. **Runtime → Change runtime type → GPU** (optional; CPU works, FairFace is slower)
2. **Run Cell 1 (Setup)** — installs deps, clones repo, mounts Drive, downloads FairFace weights
3. Edit `PANSOR_ROOT` if your Drive folder name differs
4. Run remaining cells **top to bottom**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RooneyEmily/Fitskin/blob/main/d65_fairface7_roi_walkthrough.ipynb)

### What each stage does

| Stage | Input | Output | Why |
|---|---|---|---|
| Demosaic DNGs | flash + no-flash `.DNG` | linear RGB `A0`, `B0` | colorimetry needs linear sensor data, **no camera WB** |
| Apple cheek mask | `face_landmarks.json` in zip | boolean cheek ROI | same landmarks as Hybrid D65 Colab; **no MediaPipe** |
| Reflectance | `A0`, exposure-matched `B0` | `R0 = √(A0⊙B0')` | cancels ambient×flash to scene reflectance |
| Frozen Lab | `R0` + affine + 5500K→D65 | trimmed-mean Lab | claimable color path (~5.55 ΔE) |
| FairFace-7 | 8-bit crop from demosaiced `A0` | ethnicity prior | ROI routing only (not a colorimeter) |
| Specular-tone ROI | prior + cheek Lab cloud | final Lab | deployment sampling (~3.63 ΔE) |

> **Raw vs preview:** Lab always comes from **linear RAW demosaic**. The 8-bit preview is only so FairFace sees a normal photo-like face (it was trained that way).

> **Run Cell 1 first** after every Colab session restart.

### Drive data (same as Hybrid D65 Colab)

Pansor lives in the shared FitSkin folder — **not** under `/home/...` and usually **not** as a bare `MyDrive/Pansor Dataset`.

1. Open https://drive.google.com/drive/folders/1RqbqHzTiezUAwlm9xRON0dbZAcDXn9Ep  
2. **Organize → Add shortcut to My Drive** (if you have not already)  
3. Restart runtime → re-run Cell 1  

Cell 1 searches the same candidate paths as `Pansor20_Hybrid_D65_Pipeline_Colab`.

If Drive mount cannot see **Shared with me**, run **Cell 1b** (Drive API copy).


## 0 — Setup (run this cell first every time)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# What this cell does:
#   1) install Python deps
#   2) clone / pull Fitskin from GitHub (code + calibration matrices)
#   3) mount Google Drive (Pansor DNGs + demographics xlsx)
#   4) download FairFace-7 weights once (~82 MB; not stored in git)
# ══════════════════════════════════════════════════════════════════════════════

# ── 1) Packages ───────────────────────────────────────────────────────────────
# rawpy     → demosaic DNG
# openpyxl  → demographics spreadsheet
# gdown     → FairFace weights from Google Drive
!pip install -q rawpy opencv-python-headless numpy matplotlib openpyxl gdown
try:
    import torch, torchvision  # noqa: F401  # usually already on Colab
except ImportError:
    !pip install -q torch torchvision

# ── 2) Clone / update Fitskin ─────────────────────────────────────────────────
# If the repo is private, git asks for a Personal Access Token (not your password).
import os, sys, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/RooneyEmily/Fitskin.git"  # change if using a fork
if os.path.isdir("Fitskin"):
    !cd Fitskin && git pull --ff-only || true
else:
    !git clone {REPO_URL}

REPO = Path("Fitskin").resolve()
assert (REPO / "models" / "fairface_race.py").is_file(), (
    f"Clone looks incomplete — missing models/fairface_race.py under {REPO}. "
    "Check REPO_URL / PAT, then re-run this cell."
)
assert (REPO / "scripts" / "evaluate_pansor20_chartfree_d65.py").is_file(), (
    f"Clone looks incomplete — missing pansor eval script under {REPO}."
)

# ── 3) Mount Drive (Pansor data only) ─────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# Put Fitskin first on sys.path so `import models…` resolves here
sys.path = [str(REPO)] + [p for p in sys.path if Path(p).resolve() != REPO]

import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

from delta_e_2000 import delta_e_2000
from flash_noflash_spectral import planck_xyz_y1
from models.fairface_race import FairFacePredictor, face_rgb_crop_from_landmarks
from scripts.evaluate_pansor20_chartfree_d65 import (
    D65,
    bradford_cat_matrix,
    linear_rgb_to_preview_bgr,
    load_dng_linear,
    load_apple_landmarks,
    apple_face_cheek_masks,
    match_flash_exposure,
    load_affine,
    load_demographics,
    discover_indoor_trials,
    extract_zip,
    mean_lab_on_mask,
)

# ── 4) Paths you may need to edit ─────────────────────────────────────────────
CAL_DIR = REPO / "calibration" / "tier3_affine"       # RGB→XYZ affine (frozen)
FAIRFACE_DIR = REPO / "calibration" / "fairface"      # .pt weights downloaded here
FAIRFACE_DIR.mkdir(parents=True, exist_ok=True)

# Indoor chart-free Pansor zips + demographics xlsx
# Same candidate paths as Pansor20_Hybrid_D65_Pipeline_Colab (shared FitSkin Drive).
# Optional manual override if auto-detect fails:
PANSOR_ROOT_OVERRIDE = None  # e.g. "/content/drive/MyDrive/FitSkin-RIT 2026/Emily/Pansor Dataset"

PANSOR_CANDIDATES = [
    "/content/drive/MyDrive/Shared with me/FitSkin-RIT 2026/Emily/Pansor Dataset",
    "/content/drive/Shareddrives/FitSkin-RIT 2026/Emily/Pansor Dataset",
    "/content/drive/MyDrive/FitSkin-RIT 2026/Emily/Pansor Dataset",
    "/content/drive/MyDrive/Pansor Dataset",
    "/content/drive/MyDrive/FitSkin/Pansor Dataset",
]

def _find_pansor_root():
    if PANSOR_ROOT_OVERRIDE:
        return Path(PANSOR_ROOT_OVERRIDE)
    for c in PANSOR_CANDIDATES:
        p = Path(c)
        if p.is_dir() and (
            (p / "Pansor Dataset Demographics.xlsx").is_file()
            or any(p.glob("Participant *"))
        ):
            return p
    # Shallow walk under MyDrive / Shareddrives (depth ≤ 5), like Hybrid Colab
    for base in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")):
        if not base.is_dir():
            continue
        print(f"Searching {base} for 'Pansor Dataset' …")
        base_depth = len(base.parts)
        for root, dirs, files in os.walk(base):
            depth = len(Path(root).parts) - base_depth
            if depth > 5:
                dirs.clear()
                continue
            if Path(root).name in ("Pansor Dataset", "Pansor dataset"):
                return Path(root)
    return None

PANSOR_ROOT = _find_pansor_root()
if PANSOR_ROOT is None:
    raise FileNotFoundError(
        "Cannot find Pansor Dataset on Drive.\n"
        "1) Open https://drive.google.com/drive/folders/1RqbqHzTiezUAwlm9xRON0dbZAcDXn9Ep\n"
        "2) Organize → Add shortcut to My Drive\n"
        "3) Runtime → Restart session → re-run this cell\n"
        "4) Or set PANSOR_ROOT_OVERRIDE to the folder that contains "
        "'Participant 1' … and 'Pansor Dataset Demographics.xlsx'"
    )

DEMOG_XLSX = PANSOR_ROOT / "Pansor Dataset Demographics.xlsx"
if not DEMOG_XLSX.is_file():
    hits = list(PANSOR_ROOT.glob("**/*Demographics*.xlsx"))
    if hits:
        DEMOG_XLSX = hits[0]

WORK_DIR = Path("/content/pansor_extract")              # unpacked zips (scratch)
OUT_DIR = Path("/content/d65_fairface7_roi_results")    # cohort CSV / TSV / summary
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Pansor root:", PANSOR_ROOT)
print("  demographics:", DEMOG_XLSX, "| exists:", DEMOG_XLSX.is_file())
print("  participants:", sorted(p.name for p in PANSOR_ROOT.glob("Participant *"))[:5], "…")

# ── 5) FairFace-7 weights (~82 MB, once per runtime) ──────────────────────────
FF7 = FAIRFACE_DIR / "res34_fair_align_multi_7_20190809.pt"
if not FF7.is_file():
    print("Downloading FairFace-7 weights (~82 MB)…")
    !gdown 11y0Wi3YQf21a_VcspUV4FwqzhMcfaVAB -O "{FF7}"
assert FF7.is_file(), f"Missing FairFace weights: {FF7}"

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("REPO:", REPO)
print("PANSOR_ROOT exists:", PANSOR_ROOT.is_dir(), "→", PANSOR_ROOT)
print("Setup OK.")


## 0b — Link shared Pansor via Drive API (if Cell 1 cannot see the folder)

Colab’s Drive **mount does not include “Shared with me”** unless you add a shortcut.
This cell uses the Drive **folder ID** from the Hybrid D65 Colab and copies data into
`/content/Pansor Dataset` on the runtime disk.

Use the **same Google account** in Colab that can open the shared folder in Drive.



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1b — Pull shared Pansor Dataset into /content via Drive API
# Use when MyDrive path search fails for "Shared with me".
# Folder ID matches Pansor20_Hybrid_D65_Pipeline_Colab.
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

PANSOR_FOLDER_ID = "1RqbqHzTiezUAwlm9xRON0dbZAcDXn9Ep"  # FitSkin shared Pansor Dataset
LOCAL_PANSOR = Path("/content/Pansor Dataset")
LOCAL_PANSOR.mkdir(parents=True, exist_ok=True)

drive_service = build("drive", "v3")
FOLDER_MIME = "application/vnd.google-apps.folder"

def _list_children(folder_id):
    q = f"'{folder_id}' in parents and trashed=false"
    out, token = [], None
    while True:
        resp = drive_service.files().list(
            q=q,
            spaces="drive",
            fields="nextPageToken, files(id, name, mimeType, size)",
            pageToken=token,
            pageSize=1000,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        out.extend(resp.get("files", []))
        token = resp.get("nextPageToken")
        if not token:
            break
    return out

def _download_file(file_id, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_file() and dest.stat().st_size > 0:
        return
    req = drive_service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(dest, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"  {dest.name}: {int(status.progress() * 100)}%", end="\r")
    print(f"  saved {dest}")

try:
    meta = drive_service.files().get(
        fileId=PANSOR_FOLDER_ID,
        fields="id,name",
        supportsAllDrives=True,
    ).execute()
    print("Shared folder OK:", meta.get("name"), meta.get("id"))
except Exception as e:
    raise RuntimeError(
        "Cannot open shared Pansor folder with this Google account.\n"
        "In Colab, use the same account that can open the folder in Drive.\n"
        f"Folder: https://drive.google.com/drive/folders/{PANSOR_FOLDER_ID}\n"
        f"API error: {e}"
    )

children = _list_children(PANSOR_FOLDER_ID)
print(f"{len(children)} items in shared root")
for f in sorted(children, key=lambda x: x["name"])[:15]:
    print(" ", f["name"], f["mimeType"].split(".")[-1])

def _find_xlsx(files):
    xlsx = [f for f in files if f["name"].lower().endswith((".xlsx", ".xls"))]
    preferred = [f for f in xlsx if "demograph" in f["name"].lower()]
    return preferred or xlsx

xlsx_files = _find_xlsx(children)
if not xlsx_files:
    # Search one level down (sometimes workbook sits next to participants / in Emily/)
    print("No xlsx in folder root — searching one level down…")
    for f in children:
        if f["mimeType"] != FOLDER_MIME:
            continue
        sub = _list_children(f["id"])
        hits = _find_xlsx(sub)
        if hits:
            print(f"  found under {f['name']}/")
            xlsx_files = hits
            break

print("Root sample:", [f["name"] for f in sorted(children, key=lambda x: x["name"])[:25]])
assert xlsx_files, (
    "No .xlsx found in shared Pansor folder. "
    "Open the folder in Drive and confirm 'Pansor Dataset Demographics.xlsx' is there."
)
_download_file(xlsx_files[0]["id"], LOCAL_PANSOR / xlsx_files[0]["name"])

part_folders = [
    f for f in children
    if f["mimeType"] == FOLDER_MIME and f["name"].startswith("Participant")
]
print(f"Participant folders: {len(part_folders)}")

# 0 = download all (~4.7 GB). Set to 1 for a quick single-participant demo.
LIMIT_PARTICIPANTS = 1  # start small; set 0 for full cohort later

for i, pf in enumerate(sorted(part_folders, key=lambda x: x["name"])):
    if LIMIT_PARTICIPANTS and i >= LIMIT_PARTICIPANTS:
        print(f"Stopping after {LIMIT_PARTICIPANTS} participant folder(s) — set LIMIT_PARTICIPANTS=0 for all")
        break
    dest_dir = LOCAL_PANSOR / pf["name"]
    dest_dir.mkdir(parents=True, exist_ok=True)
    zips = [f for f in _list_children(pf["id"]) if f["name"].lower().endswith(".zip")]
    print(f"{pf['name']}: {len(zips)} zip(s)")
    for z in zips:
        _download_file(z["id"], dest_dir / z["name"])

PANSOR_ROOT = LOCAL_PANSOR
DEMOG_XLSX = next(LOCAL_PANSOR.glob("*Demographics*.xlsx"))
print("\nReady.")
print("  PANSOR_ROOT =", PANSOR_ROOT)
print("  DEMOG_XLSX  =", DEMOG_XLSX)
print("Continue with Cell 2 (calibration / demographics load).")



## 1 — Load calibration, FairFace, demographics

**Frozen color path pieces (loaded once):**
- `tier3_affine` — pooled camera RGB→XYZ from chart training (not FitSkin cheeks)
- Planckian **5500 K** white → Bradford CAT → **D65** (fixed; no SCR illuminant estimate)
- FairFace-7 — race prior for ROI only
- Demographics xlsx — FitSkin Lab targets for ΔE scoring (not used at deployment)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Load the frozen color tools + FairFace + FitSkin targets
# ══════════════════════════════════════════════════════════════════════════════

# Camera RGB → XYZ affine (4×3 with bias column), from calibration/tier3_affine
M = load_affine(CAL_DIR)

# Fixed scene white for CAT: Planck locus at 5500 K, Y=1
xyz_w = planck_xyz_y1(5500.0, 0.0)
CAT = bradford_cat_matrix(xyz_w, D65)  # also applied inside mean_lab_on_mask
print("Affine M shape:", M.shape)
print("W_5500 XYZ:", np.round(xyz_w, 4))
print("D65 XYZ:", D65)

# FairFace-7 ResNet-34 (race head). Device = CUDA if available.
ff = FairFacePredictor.load(mode="7", weights_dir=FAIRFACE_DIR)
print("FairFace device:", ff.device)

# FitSkin Inside Lab targets — HARDCODED (no Drive xlsx required).
# Same values as "Pansor Dataset Demographics.xlsx" Inside columns.
PANSOR_DEMOGRAPHICS = {
    1: {"name": "Bryan", "ethnicity": "Black", "fitskin_L": 27.68, "fitskin_a": 8.79, "fitskin_b": 13.84},
    2: {"name": "Dylan", "ethnicity": "White", "fitskin_L": 61.62, "fitskin_a": 13.42, "fitskin_b": 16.5},
    3: {"name": "Eric", "ethnicity": "Asian", "fitskin_L": 55.51, "fitskin_a": 12.75, "fitskin_b": 19.13},
    4: {"name": "Luca", "ethnicity": "White", "fitskin_L": 64.62, "fitskin_a": 11.13, "fitskin_b": 16.66},
    5: {"name": "Ray", "ethnicity": "Indian", "fitskin_L": 59.63, "fitskin_a": 11.25, "fitskin_b": 17.32},
    6: {"name": "Shuyi", "ethnicity": "Asian", "fitskin_L": 59.02, "fitskin_a": 12.56, "fitskin_b": 21.19},
    7: {"name": "Sonali", "ethnicity": "Indian", "fitskin_L": 52.84, "fitskin_a": 10.91, "fitskin_b": 22.05},
    8: {"name": "Utsav", "ethnicity": "Indian", "fitskin_L": 49.65, "fitskin_a": 13.08, "fitskin_b": 21.9},
    9: {"name": "Vivian", "ethnicity": "Black", "fitskin_L": 29.64, "fitskin_a": 10.23, "fitskin_b": 15.54},
    10: {"name": "Yuan", "ethnicity": "Asian", "fitskin_L": 68.0, "fitskin_a": 9.99, "fitskin_b": 15.59},
    11: {"name": "Zeevan", "ethnicity": "Indian", "fitskin_L": 50.25, "fitskin_a": 13.49, "fitskin_b": 23.47},
    12: {"name": "Sanaz", "ethnicity": "Iranian", "fitskin_L": 58.85, "fitskin_a": 12.17, "fitskin_b": 19.64},
    13: {"name": "Brendan", "ethnicity": "White", "fitskin_L": 60.53, "fitskin_a": 14.55, "fitskin_b": 17.56},
    14: {"name": "Raees", "ethnicity": "Indian", "fitskin_L": 48.47, "fitskin_a": 12.17, "fitskin_b": 21.97},
    15: {"name": "Nima", "ethnicity": "Iranian", "fitskin_L": 58.01, "fitskin_a": 12.11, "fitskin_b": 18.26},
    16: {"name": "Nick", "ethnicity": "White", "fitskin_L": 63.14, "fitskin_a": 12.31, "fitskin_b": 17.27},
    17: {"name": "Jalona", "ethnicity": "Black", "fitskin_L": 41.64, "fitskin_a": 11.72, "fitskin_b": 21.64},
    18: {"name": "Gabe", "ethnicity": "White", "fitskin_L": 65.26, "fitskin_a": 11.88, "fitskin_b": 15.65},
    19: {"name": "Chidera", "ethnicity": "Black", "fitskin_L": 29.64, "fitskin_a": 10.23, "fitskin_b": 15.45},
    20: {"name": "Charles", "ethnicity": "Asian", "fitskin_L": 59.82, "fitskin_a": 11.59, "fitskin_b": 21.14},
}

demo = PANSOR_DEMOGRAPHICS
print(f"Demographics: {len(demo)} participants (hardcoded Inside Lab)")
print("Example P1:", demo.get(1))

# Data root for zips: prefer /content copy from Drive API, else prior PANSOR_ROOT
LOCAL_PANSOR = Path("/content/Pansor Dataset")
if LOCAL_PANSOR.is_dir() and any(LOCAL_PANSOR.glob("Participant *")):
    PANSOR_ROOT = LOCAL_PANSOR
elif "PANSOR_ROOT" not in globals() or PANSOR_ROOT is None or not Path(PANSOR_ROOT).is_dir():
    PANSOR_ROOT = LOCAL_PANSOR
print("PANSOR_ROOT =", PANSOR_ROOT, "| exists:", Path(PANSOR_ROOT).is_dir())


## 2 — Single-trial walkthrough

Pick one indoor zip (no bag / outside / light-box in the name).  
Cheek geometry comes from **Apple Vision** `face_landmarks.json` inside the zip.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Choose one indoor chart-free trial and unpack its zip
# Zip contents we need:
#   • no-flash DNG, flash DNG
#   • face_landmarks.json  (Apple Vision, full-res coordinates)
# ══════════════════════════════════════════════════════════════════════════════

trials = discover_indoor_trials(PANSOR_ROOT)
print(f"Indoor chart-free trials found: {len(trials)}")
assert len(trials) > 0, "No indoor zips found — check PANSOR_ROOT"

# Default = first trial. Uncomment an override to pick someone specific:
PARTICIPANT_ID = int(trials[0]["participant_id"])
TRIAL = int(trials[0]["trial"])
# PARTICIPANT_ID, TRIAL = 6, 1   # Shuyi
# PARTICIPANT_ID, TRIAL = 1, 1   # Bryan

t = next(
    x for x in trials
    if int(x["participant_id"]) == PARTICIPANT_ID and int(x["trial"]) == TRIAL
)
meta = demo[PARTICIPANT_ID]
print("Selected:", t["subject_id"], meta["name"], meta["ethnicity"])
print("Zip:", t["zip_path"])
print("FitSkin Lab (target):", meta["fitskin_L"], meta["fitskin_a"], meta["fitskin_b"])

ZIP_PATH = Path(t["zip_path"])
work = WORK_DIR / t["subject_id"]
nf, fl, lm_path = extract_zip(ZIP_PATH, work)
print("Extracted files:", nf.name, "|", fl.name, "|", lm_path.name)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — RAW demosaic → cheek mask → flash/no-flash reflectance
#
# Colorimetry path (linear RAW):
#   A0, B0  = demosaic WITHOUT camera white-balance
#   cheek   = Apple landmark polygons scaled to demosaic resolution
#   B0m     = flash frame exposure-matched to no-flash on the cheek
#   R0      = √(A0 ⊙ B0m)   ≈ reflectance (ambient×flash cancel)
#
# Preview path (display only / FairFace input):
#   8-bit stretch of A0 — NOT used for Lab
# ══════════════════════════════════════════════════════════════════════════════

# half_size=True → faster / less RAM on Colab; still linear RAW
A0 = load_dng_linear(nf, half_size=True, use_camera_wb=False)
B0 = load_dng_linear(fl, half_size=True, use_camera_wb=False)
if B0.shape != A0.shape:
    B0 = cv2.resize(B0, (A0.shape[1], A0.shape[0]), interpolation=cv2.INTER_AREA)

# Landmarks are stored at full capture size; helper rescales polygons to A0 shape
lm = load_apple_landmarks(lm_path)
_, cheek = apple_face_cheek_masks(lm, A0.shape[0], A0.shape[1])
print("Demosaic shape:", A0.shape, "| cheek pixels:", int(np.count_nonzero(cheek)))

# Match mean cheek luma of flash to no-flash, then geometric-mean reflectance
B0m, s = match_flash_exposure(A0, B0, cheek)
R0 = np.sqrt(np.maximum(A0, 0) * np.maximum(B0m, 0) + 1e-8)
print(f"Flash exposure scale s={s:.4f}  (B0' = s · B0)")

# 8-bit preview from linear A0 (FairFace + plots only)
preview = linear_rgb_to_preview_bgr(A0)
overlay = preview.copy()
overlay[cheek > 0] = (0.6 * overlay[cheek > 0] + 0.4 * np.array([0, 255, 0])).astype(np.uint8)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
ax[0].set_title("No-flash preview (from RAW)"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
ax[1].set_title("Apple cheek mask"); ax[1].axis("off")
ax[2].imshow(np.clip(R0 ** (1 / 2.2), 0, 1))
ax[2].set_title("Reflectance R0 (γ for display)"); ax[2].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Frozen color path (claimable): trimmed-mean cheek Lab
#   R0 → affine RGB→XYZ → Bradford CAT (5500K→D65) → CIELAB
#   l_sampling="off" → 5% trimmed mean on cheek (no ethnicity / FairFace)
# Compare to FitSkin Inside Lab with CIEDE2000.
# ══════════════════════════════════════════════════════════════════════════════

fit = np.array([meta["fitskin_L"], meta["fitskin_a"], meta["fitskin_b"]], dtype=np.float64)

Lab_frozen, _ = mean_lab_on_mask(
    R0, cheek, M,
    xyz_scene_white=xyz_w,   # triggers 5500K→D65 CAT inside
    cat_degree=1.0,
    l_sampling="off",        # trimmed mean only
)
de_frozen = float(delta_e_2000(Lab_frozen, fit))
print(f"Frozen Lab = ({Lab_frozen[0]:.1f}, {Lab_frozen[1]:.1f}, {Lab_frozen[2]:.1f})")
print(f"FitSkin    = ({fit[0]:.1f}, {fit[1]:.1f}, {fit[2]:.1f})")
print(f"ΔE00 frozen (no ROI heuristic) = {de_frozen:.2f}")
print("(Cohort mean for this path is ~5.55)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Deployment ROI: FairFace-7 prior → specular_tone sampling
#
# FairFace sees an 8-bit face crop (from RAW demosaic preview + Apple landmarks).
# It does NOT change XYZ/Lab math — only which cheek pixels / L* rule we use.
#
# Mapping (FairFace → Pansor ROI key):
#   Black/White/Indian as-is
#   East/SE Asian → Asian
#   Middle Eastern → Iranian
#   Latino_Hispanic → Asian
# ══════════════════════════════════════════════════════════════════════════════

face_rgb = face_rgb_crop_from_landmarks(preview, lm, padding=0.35)
ff_out = ff.predict_rgb(face_rgb)

print("FairFace-7 raw label:", ff_out["fairface_label"], f"(conf={ff_out['confidence']:.2f})")
print("→ ROI ethnicity key: ", ff_out["predicted_ethnicity"])
print("Class probs:", {k: round(v, 3) for k, v in ff_out["race_probs"].items()})

plt.figure(figsize=(3, 3))
plt.imshow(face_rgb)
plt.title(f"FF crop → {ff_out['fairface_label']}")
plt.axis("off")
plt.show()

# Same color math as Cell 5, but cheek aggregation follows specular_tone rules
# conditioned on the FairFace prior (deployment — no demographics labels).
Lab_ff, sm = mean_lab_on_mask(
    R0, cheek, M,
    xyz_scene_white=xyz_w,
    cat_degree=1.0,
    l_sampling="specular_tone",
    ethnicity=ff_out["predicted_ethnicity"],
)
de_ff = float(delta_e_2000(Lab_ff, fit))

# Oracle: same ROI rules but with true demographics ethnicity (not available at deploy)
Lab_oracle, _ = mean_lab_on_mask(
    R0, cheek, M,
    xyz_scene_white=xyz_w,
    cat_degree=1.0,
    l_sampling="specular_tone",
    ethnicity=meta["ethnicity"],
)
de_oracle = float(delta_e_2000(Lab_oracle, fit))

print(f"\nD65-FairFace7-ROI = ({Lab_ff[0]:.1f}, {Lab_ff[1]:.1f}, {Lab_ff[2]:.1f})  ΔE00={de_ff:.2f}")
print(f"Oracle demographics= ({Lab_oracle[0]:.1f}, {Lab_oracle[1]:.1f}, {Lab_oracle[2]:.1f})  ΔE00={de_oracle:.2f}")
print(f"Frozen trimmed mean= ({Lab_frozen[0]:.1f}, {Lab_frozen[1]:.1f}, {Lab_frozen[2]:.1f})  ΔE00={de_frozen:.2f}")
print("Which specular_tone branch fired:", sm)


## 3 — Full indoor chart-free cohort (optional)

Runs the production CLI on all indoor zips (`N ≈ 65`). Writes Emily-format TSV + `summary.json`.

Cell 8 plots ΔE₀₀ histograms **by ethnicity** (Hybrid Colab style) from the CSV.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Full cohort eval (D65-FairFace7-ROI)
# Writes demographics xlsx from the hardcoded table (no Drive workbook needed),
# then runs the production evaluator on whatever zips are under PANSOR_ROOT.
# Tip: only Participant 1 downloaded ⇒ small n; download more zips for full cohort.
# ══════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import subprocess, json
from openpyxl import Workbook

PANSOR_ROOT = Path("/content/Pansor Dataset")
WORK_DIR = Path("/content/pansor_extract")
OUT_DIR = Path("/content/d65_fairface7_roi_results")
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Materialize demographics workbook for the CLI (Inside Lab columns)
DEMOG_XLSX = PANSOR_ROOT / "Pansor Dataset Demographics.xlsx"
wb = Workbook()
ws = wb.active
ws.append(["Pansor Dataset Demographics"])
ws.append(["Participant ID", "Name", "Ethnicity", "L*", "a*", "b*"])
for pid, meta in sorted(demo.items()):
    ws.append([
        pid, meta["name"], meta["ethnicity"],
        meta["fitskin_L"], meta["fitskin_a"], meta["fitskin_b"],
    ])
wb.save(DEMOG_XLSX)
print("Wrote", DEMOG_XLSX)
print("Zips available:", len(list(PANSOR_ROOT.glob("Participant */*.zip"))))

cmd = [
    "python3", str(REPO / "scripts" / "evaluate_pansor20_chartfree_d65.py"),
    "--data-root", str(PANSOR_ROOT),
    "--demographics", str(DEMOG_XLSX),
    "--cal-dir", str(CAL_DIR),
    "--scr-mode", "preawb_cat",
    "--fixed-cat-k", "5500",
    "--l-sampling", "fairface7",
    "--fairface-dir", str(FAIRFACE_DIR),
    "--emily-tsv",
    "--work-dir", str(WORK_DIR),
    "--out-dir", str(OUT_DIR),
]
print(" ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"evaluator failed with exit {proc.returncode}")

summary = json.loads((OUT_DIR / "summary.json").read_text())
print("\n=== D65-FairFace7-ROI ===")
print(f"n={summary['n_trials']}  mean={summary['mean_de00']:.2f}  median={summary['median_de00']:.2f}")
print(f"{'Ethnicity':10s} {'n':>4s} {'mean':>8s} {'median':>8s}")
for eth, st in summary["by_ethnicity"].items():
    print(f"{eth:10s} {st['n']:4d} {st['mean_de00']:8.2f} {st['median_de00']:8.2f}")
print("Emily TSV:", OUT_DIR / "table_emily_format.tsv")



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Plot ΔE00 by ethnicity (same style as Hybrid D65 Colab)
# Reads the CSV written by Cell 7. With only Participant 1 downloaded you will
# mostly see Black; download more zips for the full demographic breakdown.
# ══════════════════════════════════════════════════════════════════════════════
from pathlib import Path
from statistics import mean, median
import csv
import numpy as np
import matplotlib.pyplot as plt

OUT_DIR = Path("/content/d65_fairface7_roi_results")
csv_path = OUT_DIR / "pansor20_chartfree_d65.csv"
if not csv_path.is_file():
    # fallback name variants
    hits = list(OUT_DIR.glob("*.csv"))
    assert hits, f"No CSV under {OUT_DIR} — run Cell 7 first"
    csv_path = hits[0]

rows = list(csv.DictReader(csv_path.open()))
print(f"Loaded {len(rows)} trials from {csv_path.name}")

by_eth = {}
for r in rows:
    eth = (r.get("ethnicity") or "Unknown").strip()
    by_eth.setdefault(eth, []).append(float(r["de00"]))

order = [e for e in ["Black", "Indian", "Asian", "Iranian", "White"] if e in by_eth]
order += [e for e in sorted(by_eth) if e not in order]
colors = {
    "Black": "#2c3e50", "Indian": "#c0392b", "Asian": "#2980b9",
    "Iranian": "#16a085", "White": "#d4a017",
}

# --- Histograms by ethnicity ---
n_panels = max(len(order), 1)
ncols = 3
nrows = int(np.ceil(n_panels / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(11, 3.2 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
vmax = max((max(v) for v in by_eth.values()), default=10)
bins = np.arange(0, max(16.5, np.ceil(vmax) + 1.5), 1.0)

for i, eth in enumerate(order):
    ax = axes[i]
    vals = by_eth[eth]
    ax.hist(vals, bins=bins, color=colors.get(eth, "#7f8c8d"), edgecolor="white", alpha=0.9)
    ax.axvline(median(vals), color="#e74c3c", ls="--", lw=1.5, label=f"med={median(vals):.2f}")
    ax.axvline(5.55, color="#27ae60", ls=":", lw=1.2, alpha=0.8, label="frozen 5.55")
    ax.set_title(f"{eth} (n={len(vals)})")
    ax.set_xlabel(r"$\Delta E_{00}$")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8, frameon=False)
for j in range(len(order), len(axes)):
    axes[j].axis("off")
fig.suptitle(r"D65-FairFace7-ROI: $\Delta E_{00}$ vs FitSkin by ethnicity", fontsize=13)
plt.show()

# --- Mean / median bar chart ---
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(order))
means = [mean(by_eth[e]) for e in order]
meds = [median(by_eth[e]) for e in order]
w = 0.35
ax.bar(x - w / 2, means, w, label="mean", color="#34495e")
ax.bar(x + w / 2, meds, w, label="median", color="#e67e22")
ax.axhline(5.55, color="#27ae60", ls="--", label="frozen preawb 5.55")
ax.axhline(5.93, color="#8e44ad", ls=":", label="camera-WB ref 5.93")
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_ylabel(r"$\Delta E_{00}$")
ax.set_title("Mean / median ΔE00 by ethnicity")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

# --- Table ---
print(f"{'Ethnicity':10s} {'n':>4s} {'median':>8s} {'mean':>8s}")
all_v = []
for eth in order:
    v = by_eth[eth]
    all_v.extend(v)
    print(f"{eth:10s} {len(v):4d} {median(v):8.2f} {mean(v):8.2f}")
print(f"{'ALL':10s} {len(all_v):4d} {median(all_v):8.2f} {mean(all_v):8.2f}")

# Optional download
try:
    from google.colab import files
    tsv = OUT_DIR / "table_emily_format.tsv"
    if tsv.is_file():
        files.download(str(tsv))
except Exception:
    pass



## 4 — Reference numbers (pinned)

| Method | Runtime labels? | Mean ΔE₀₀ |
|---|---|---:|
| Camera-WB + affine gate | — | ~5.93 |
| Frozen `preawb_cat` (sampling off) | — | **5.55** |
| Lab tone→ethnicity ROI (LOSO) | no | 3.89 |
| **D65-FairFace7-ROI** | **no** | **3.63** |
| Demographics + ROI (oracle) | yes | 3.23 |

### CLI (local)
```bash
python3 scripts/evaluate_pansor20_chartfree_d65.py \
  --scr-mode preawb_cat --fixed-cat-k 5500 \
  --l-sampling fairface7 --emily-tsv \
  --out-dir results/pansor20_fairface7
```

Methods write-up: `docs/d65_fairface7_roi_overleaf.tex`

### Drive tip (from Hybrid D65 Colab)
If Pansor is missing after mount, open  
https://drive.google.com/drive/folders/1RqbqHzTiezUAwlm9xRON0dbZAcDXn9Ep  
→ **Organize → Add shortcut to My Drive**, restart runtime, re-run Cell 1.
